In [1]:
import pprint
import ollama
import json
import time

import pandas as pd

In [2]:
df = pd.read_excel('comments_w_analysis_exclude_place_comments_clean.xlsx', usecols=range(10))

In [3]:
print(f"Total number of samples: {len(df)}")
print(f"Number of null samples: {df['comment_EN'].isnull().sum()}")

# Filter data for usable comments
df = df[df['comment_EN'].notnull()].copy()
print(f"Number of usable samples: {len(df)}")

Total number of samples: 733
Number of null samples: 0
Number of usable samples: 733


In [4]:
from pydantic import BaseModel

class Sentiment(BaseModel):
  comment_id: int
  sentiment_score: float | None 
  confidence_score: float | None
  reasoning: str | None
    
class SentimentList(BaseModel):
  sentiments: list[Sentiment]

In [6]:
def ollama_batch_inference_with_reasoning(df: pd.DataFrame, prompt: str, comment_col: str, batch_size: int = 10, model: str = 'deepseek-r1:8b', output_name: str = 'prague_sentiment_results_with_reasoning.csv', verbose: bool = True):
    """ Run inference by batches with ollama

    Args:
        df (pd.DataFrame): Dataframe containing comments.
        prompt (str): Prompt for the LLM.
        comment_col (str): Name of the DataFrame column where the comments to analyse are located.
        batch_size (int, optional): Number of comments to include in the LLM prompt at a time. Defaults to 10.
        model (str, optional): Ollama model to use. Defaults to 'deepseek-r1:8b'.
        output_name (str, optional): Name for the output file containing the sentiment analysis results. Defaults to 'prague_sentiment_results_ollama_with_reasoning.csv'.
        verbose (bool, optional): Option to print information messages. Can be True or False. Defaults to True.
    """    
        
    # Batch processing for sentiment analysis
    count = 0
    results_df = pd.DataFrame()

    for start in range(0, len(df), batch_size):
        end = start + batch_size
        batch_df = df.iloc[start:end].copy()
        batch_comments = batch_df[['ID', comment_col]].values.tolist()
        
        new_prompt = prompt[:]
        for id, comment in batch_comments:
            new_prompt += f"{id}: '{comment}'\n"
            
        response = ollama.generate(
            model=model,
            prompt=new_prompt,
            format=SentimentList.model_json_schema(),
        )
        if verbose:
            print(f"### Input ###\n{new_prompt}### END Input ###\n")
            print(f"### LLM output ###\n{response.response}### END LLM output ###\n")
        try:
            parsed_response = SentimentList.model_validate_json(response.response).sentiments
            count += len(parsed_response)
            
            if verbose:
                print(f"Number of texts: {len(batch_comments)}")
                print(f"Number of answers: {len(parsed_response)}")
                print(f"### LLM parsed output ###\n")
                pprint.pprint(parsed_response)
                print(f"\n### END LLM parsed output ###\n")
            
            ids = []
            labels = []
            scores = []
            reasonings = []
            
            for element in parsed_response:
                sentiment = element.sentiment_score
                if sentiment is not None:
                    if sentiment == 1.0:
                        sentiment = "positive"
                    elif sentiment == -1.0:
                        sentiment = "negative"
                    elif sentiment == 0:
                        sentiment = "neutral"
                else:
                    sentiment = "none"
            
                ids.append(element.comment_id)
                labels.append(sentiment)
                scores.append(element.confidence_score)
                reasonings.append(element.reasoning)
            
            partial_result = pd.merge(batch_df, pd.DataFrame({"ID": ids, "sentiment_label": labels, "confidence_score": scores, "reasoning": reasonings}), on="ID", how="left")
            results_df = pd.concat([results_df, partial_result], ignore_index=True)
            if 'index' in results_df:
                results_df.drop(columns="index", inplace=True)
            results_df.to_csv(output_name, index=False)
            
            print(f"Processed {count}/{len(df)} comments\n")  # Progress update
            
        except Exception as err:
            print(f'Could not parse responses from comments {start} to {end}')
            print(f'{err=}\n')

    return results_df


# English

In [7]:
comment_col = 'comment_EN'

system_prompt = """
You are a sentiment analyser system of Public Participation Geographic Information Systems (PPGIS) data.
The data is extracted from citizen comments about the city of Prague, translated from Czech to English.
Your task is to analyse and predict the sentiment of the comments, as well as provide a confidence score of the prediction and your reasoning for it.

### 1. SENTIMENT FORMATTING RULES
- If a comment expresses no clear sentiment, assign a neutral sentiment score.
- Do not invent or lie if you do not know the answer; say None if you do not know.
- Note that the comments may have street names and variability of PPGIS data.

### 2. OUTPUT FORMAT
The output should be a list of sentiment objects, where each input comment is given a sentiment analysis. The fields of a sentiment object are:
- comment_id: The ID of the input comment to analyse.
- sentiment_score: The numerical sentiment score is -1 for negative sentiment, 1 for positive sentiment, 0 for neutral sentiment, or None if no sentiment can be assigned.
- confidence_score: The confidence score of your prediction.
- reasoning: A reasoning of why you assigned this sentiment to this comment.

### 3. FEW-SHOT EXAMPLES (Do not confuse these with your real task)
Input: [
    {"comment_id": 1, "comment": "The new pedestrian zone is absolutely wonderful, such a joy to walk through!"},
    {"comment_id": 2, "comment": "The old bus routes were much better, this new schedule is a complete disaster for commuters."},
    {"comment_id": 3, "comment": "They are planning to build a new roundabout."}
]

Output: {
  "sentiments": [
    {
      "comment_id": 1,
      "sentiment_score": 1,
      "confidence_score": 0.9,
      "reasoning": "The words 'absolutely' and 'wonderful' indicate a strong positive sentiment, so the comment is considered to be positive."
    },
    {
      "comment_id": 2,
      "sentiment_score": -1,
      "confidence_score": 0.9,
      "reasoning": "Phrases 'old bus routes were much better' and a 'complete disaster' clearly point to strong negative sentiment regarding the new bus schedule."
    },
    {
      "comment_id": 3,
      "sentiment_score": 0,
      "confidence_score": 0.8,
      "reasoning": "The comment simply states a factual plan without expressing any opinion or emotion."
    }    
  ]
}

### END OF EXAMPLES
Now, analyze the following comments based strictly on the text provided below:
"""

In [8]:
# query_data = df.copy() # Whole dataset
query_data = df.iloc[:20].copy() # First 20 comments

In [9]:
start_time = time.time()

results_df = ollama_batch_inference_with_reasoning(
    query_data,
    prompt=system_prompt,
    comment_col=comment_col,
    batch_size=10,
    model='deepseek-r1:8b',
    output_name='prague_sentiment_results_with_reasoning_EN_1.csv',
    verbose=True
)

run_time = time.time() - start_time

print(f"Sentiment analysis for comments in English completed in {run_time} seconds.")

### Input ###

You are a sentiment analyser system of Public Participation Geographic Information Systems (PPGIS) data.
The data is extracted from citizen comments about the city of Prague, translated from Czech to English.
Your task is to analyse and predict the sentiment of the comments, as well as provide a confidence score of the prediction and your reasoning for it.

### 1. SENTIMENT FORMATTING RULES
- If a comment expresses no clear sentiment, assign a neutral sentiment score.
- Do not invent or lie if you do not know the answer; say None if you do not know.
- Note that the comments may have street names and variability of PPGIS data.

### 2. OUTPUT FORMAT
The output should be a list of sentiment objects, where each input comment is given a sentiment analysis. The fields of a sentiment object are:
- comment_id: The ID of the input comment to analyse.
- sentiment_score: The numerical sentiment score is -1 for negative sentiment, 1 for positive sentiment, 0 for neutral sentiment, o

In [9]:
start_time = time.time()

results_df = ollama_batch_inference_with_reasoning(
    query_data,
    prompt=system_prompt,
    comment_col=comment_col,
    batch_size=10,
    model='deepseek-r1:8b',
    output_name='prague_sentiment_results_with_reasoning_EN_2.csv',
    verbose=True
)

run_time = time.time() - start_time

print(f"Sentiment analysis for comments in English completed in {run_time} seconds.")

### Input ###

You are a sentiment analyser system of Public Participation Geographic Information Systems (PPGIS) data.
The data is extracted from citizen comments about the city of Prague, translated from Czech to English.
Your task is to analyse and predict the sentiment of the comments, as well as provide a confidence score of the prediction and your reasoning for it.

### 1. SENTIMENT FORMATTING RULES
- If a comment expresses no clear sentiment, assign a neutral sentiment score.
- Do not invent or lie if you do not know the answer; say None if you do not know.
- Note that the comments may have street names and variability of PPGIS data.

### 2. OUTPUT FORMAT
The output should be a list of sentiment objects, where each input comment is given a sentiment analysis. The fields of a sentiment object are:
- comment_id: The ID of the input comment to analyse.
- sentiment_score: The numerical sentiment score (between -1 and 1), or None if no sentiment can be assigned.
- confidence_score: T

In [10]:
start_time = time.time()

results_df = ollama_batch_inference_with_reasoning(
    query_data,
    prompt=system_prompt,
    comment_col=comment_col,
    batch_size=10,
    model='deepseek-r1:8b',
    output_name='prague_sentiment_results_with_reasoning_EN_3.csv',
    verbose=True
)

run_time = time.time() - start_time

print(f"Sentiment analysis for comments in English completed in {run_time} seconds.")

### Input ###

You are a sentiment analyser system of Public Participation Geographic Information Systems (PPGIS) data.
The data is extracted from citizen comments about the city of Prague, translated from Czech to English.
Your task is to analyse and predict the sentiment of the comments, as well as provide a confidence score of the prediction and your reasoning for it.

### 1. SENTIMENT FORMATTING RULES
- If a comment expresses no clear sentiment, assign a neutral sentiment score.
- Do not invent or lie if you do not know the answer; say None if you do not know.
- Note that the comments may have street names and variability of PPGIS data.

### 2. OUTPUT FORMAT
The output should be a list of sentiment objects, where each input comment is given a sentiment analysis. The fields of a sentiment object are:
- comment_id: The ID of the input comment to analyse.
- sentiment_score: The numerical sentiment score (between -1 and 1), or None if no sentiment can be assigned.
- confidence_score: T

In [11]:
start_time = time.time()

results_df = ollama_batch_inference_with_reasoning(
    query_data,
    prompt=system_prompt,
    comment_col=comment_col,
    batch_size=10,
    model='deepseek-r1:8b',
    output_name='prague_sentiment_results_with_reasoning_EN_4.csv',
    verbose=True
)

run_time = time.time() - start_time

print(f"Sentiment analysis for comments in English completed in {run_time} seconds.")

### Input ###

You are a sentiment analyser system of Public Participation Geographic Information Systems (PPGIS) data.
The data is extracted from citizen comments about the city of Prague, translated from Czech to English.
Your task is to analyse and predict the sentiment of the comments, as well as provide a confidence score of the prediction and your reasoning for it.

### 1. SENTIMENT FORMATTING RULES
- If a comment expresses no clear sentiment, assign a neutral sentiment score.
- Do not invent or lie if you do not know the answer; say None if you do not know.
- Note that the comments may have street names and variability of PPGIS data.

### 2. OUTPUT FORMAT
The output should be a list of sentiment objects, where each input comment is given a sentiment analysis. The fields of a sentiment object are:
- comment_id: The ID of the input comment to analyse.
- sentiment_score: The numerical sentiment score (between -1 and 1), or None if no sentiment can be assigned.
- confidence_score: T

In [12]:
start_time = time.time()

results_df = ollama_batch_inference_with_reasoning(
    query_data,
    prompt=system_prompt,
    comment_col=comment_col,
    batch_size=10,
    model='deepseek-r1:8b',
    output_name='prague_sentiment_results_with_reasoning_EN_5.csv',
    verbose=True
)

run_time = time.time() - start_time

print(f"Sentiment analysis for comments in English completed in {run_time} seconds.")

### Input ###

You are a sentiment analyser system of Public Participation Geographic Information Systems (PPGIS) data.
The data is extracted from citizen comments about the city of Prague, translated from Czech to English.
Your task is to analyse and predict the sentiment of the comments, as well as provide a confidence score of the prediction and your reasoning for it.

### 1. SENTIMENT FORMATTING RULES
- If a comment expresses no clear sentiment, assign a neutral sentiment score.
- Do not invent or lie if you do not know the answer; say None if you do not know.
- Note that the comments may have street names and variability of PPGIS data.

### 2. OUTPUT FORMAT
The output should be a list of sentiment objects, where each input comment is given a sentiment analysis. The fields of a sentiment object are:
- comment_id: The ID of the input comment to analyse.
- sentiment_score: The numerical sentiment score (between -1 and 1), or None if no sentiment can be assigned.
- confidence_score: T

# Czech

In [13]:
# Filter data for usable comments
df = df[df['comment_CZ'].notnull()].copy()
print(f"Number of usable samples: {len(df)}")

Number of usable samples: 733


In [14]:
comment_col = 'comment_CZ'

system_prompt = """
Jste systém pro analýzu sentimentu dat z geografických informačních systémů pro veřejnou participaci (PPGIS).
Data jsou extrahována z komentářů občanů o městě Praze v původním českém jazyce.
Vaším úkolem je analyzovat a předpovědět sentiment komentářů, poskytnout skóre spolehlivosti předpovědi a zdůvodnění vaší předpovědi.

### 1. PRAVIDLA FORMÁTOVÁNÍ SENTIMENTU
- Pokud komentář nevyjadřuje jasný názor, přiřaďte mu neutrální hodnocení.
- Nevymýšlejte si a nelžete, pokud neznáte odpověď; pokud nevíte, uveďte "Žádný".
- Upozorňujeme, že komentáře mohou obsahovat názvy ulic a variabilitu dat PPGIS.

### 2. VÝSTUPNÍ FORMÁT
Výstupem by měl být seznam objektů sentimentu, kde každý vstupní komentář je opatřen analýzou sentimentu. Pole objektu sentimentu jsou:
- comment_id: ID vstupního komentáře, který má být analyzován.
- sentiment_score: Číselné hodnocení sentimentu je -1 pro negativní sentiment, 1 pro pozitivní sentiment, 0 pro neutrální sentiment nebo Žádné, pokud nelze přiřadit žádný sentiment.
- confidence_score: Skóre spolehlivosti vaší předpovědi.
- reasoning: Odůvodnění, proč jste tomuto komentáři přiřadili tento sentiment.


### 3. PŘÍKLADY NĚKOLIKA ZÁBĚRŮ (Nezaměňujte je se svým skutečným úkolem)
Vstup: [
    {"comment_id": 1, "comment": "Nová pěší zóna je naprosto úžasná, je radost se po ní procházet!"},
    {"comment_id": 2, "comment": "Staré autobusové linky byly mnohem lepší, nový jízdní řád je pro dojíždějící naprostá katastrofa."},
    {"comment_id": 3, "comment": "Plánují výstavbu nového kruhového objezdu."}
]

Výstup: {
  "sentiments": [
    {
      "comment_id": 1,
      "sentiment_score": 1,
      "confidence_score": 0.9,
      "reasoning": "Slova „absolutně“ a „úžasné“ vyjadřují silný pozitivní sentiment, takže komentář je považován za pozitivní."
    },
    {
      "comment_id": 2,
      "sentiment_score": -1,
      "confidence_score": 0.9,
      "reasoning": "Fráze „staré autobusové linky byly mnohem lepší“ a „naprostá katastrofa“ jasně poukazují na silně negativní názor na nový jízdní řád autobusů."
    },
    {
      "comment_id": 3,
      "sentiment_score": 0,
      "confidence_score": 0.8,
      "reasoning": "Komentář pouze uvádí faktický plán, aniž by vyjadřoval jakýkoli názor nebo emoce."
    }    
  ]
}

### KONEC PŘÍKLADŮ
Nyní analyzujte následující komentáře výhradně na základě níže uvedeného textu:
"""

In [15]:
query_data = df.copy() # Whole dataset

In [ ]:
start_time = time.time()

results_df = ollama_batch_inference_with_reasoning(
    query_data,
    prompt=system_prompt,
    comment_col=comment_col,
    batch_size=10,
    model='deepseek-r1:8b',
    output_name='prague_sentiment_results_with_reasoning_CZ_deepseek8b_1.csv',
    verbose=True
)

run_time = time.time() - start_time

print(f"Sentiment analysis for comments in Czech completed in {run_time} seconds.")

In [ ]:
start_time = time.time()

results_df = ollama_batch_inference_with_reasoning(
    query_data,
    prompt=system_prompt,
    comment_col=comment_col,
    batch_size=10,
    model='deepseek-r1:8b',
    output_name='prague_sentiment_results_with_reasoning_CZ_deepseek8b_2.csv',
    verbose=True
)

run_time = time.time() - start_time

print(f"Sentiment analysis for comments in Czech completed in {run_time} seconds.")

In [ ]:
start_time = time.time()

results_df = ollama_batch_inference_with_reasoning(
    query_data,
    prompt=system_prompt,
    comment_col=comment_col,
    batch_size=10,
    model='deepseek-r1:8b',
    output_name='prague_sentiment_results_with_reasoning_CZ_deepseek8b_3.csv',
    verbose=True
)

run_time = time.time() - start_time

print(f"Sentiment analysis for comments in Czech completed in {run_time} seconds.")

In [ ]:
start_time = time.time()

results_df = ollama_batch_inference_with_reasoning(
    query_data,
    prompt=system_prompt,
    comment_col=comment_col,
    batch_size=10,
    model='deepseek-r1:8b',
    output_name='prague_sentiment_results_with_reasoning_CZ_deepseek8b_4.csv',
    verbose=True
)

run_time = time.time() - start_time

print(f"Sentiment analysis for comments in Czech completed in {run_time} seconds.")

In [ ]:
start_time = time.time()

results_df = ollama_batch_inference_with_reasoning(
    query_data,
    prompt=system_prompt,
    comment_col=comment_col,
    batch_size=10,
    model='deepseek-r1:8b',
    output_name='prague_sentiment_results_with_reasoning_CZ_deepseek8b_5.csv',
    verbose=True
)

run_time = time.time() - start_time

print(f"Sentiment analysis for comments in Czech completed in {run_time} seconds.")